# 00 - Data preparation and EDA

This notebook builds the reproducible EEG window dataset from the repository's raw CSV files and validated `data/Annotations.xlsx`. It never substitutes provisional `h` annotations silently.

The shared split is group-isolated (patient when possible), approximately 64%/16%/20%, and is reused by every model notebook.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'eeg_app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from eeg_app.src.experiments import prepare_data, load_prepared, validate_dataset, feature_names

prepared = prepare_data(ROOT, seed=42)
print('split group:', prepared['split_group'])
print('warnings:', len(prepared['warnings']))
for warning in prepared['warnings']:
    print('WARNING:', warning)

In [ ]:
dataset = load_prepared(ROOT)
report = validate_dataset(dataset)
display(pd.Series(report, dtype='object'))
display(dataset.groupby('split')['label'].agg(['count', 'mean']))
display(dataset.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dataset['label'].value_counts().sort_index().plot.bar(ax=axes[0], title='Window class counts')
dataset.groupby('split')['label'].value_counts().unstack(fill_value=0).plot.bar(ax=axes[1], title='Class counts by shared split')
plt.tight_layout()

In [ ]:
features = feature_names(dataset)
numeric = dataset[features].apply(pd.to_numeric, errors='coerce')
class_summary = dataset.groupby('label')[features].median().T
display(class_summary.head(30))

top = numeric.corrwith(dataset['label']).abs().sort_values(ascending=False).head(20)
display(top.rename('absolute_point_biserial_proxy').to_frame())
plt.figure(figsize=(12, 8))
numeric[top.index[:12]].corr().style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation among highest label-associated features')
plt.show()
print('Interpretation: Mean/Min/Max/Amplitude/RMS/Energy are mathematically related; high within-channel correlation indicates redundancy. Frequency bands are raw-sample Welch powers and should be interpreted with the temporal features, not as independent evidence.')

## Reproducibility and limitations

- The exact split is saved in `extracted_data/window_features_with_split.parquet` and `split_assignments.json`.
- `train_none`, `train_undersample`, `train_oversample`, and `train_hybrid` contain training data only. Validation and test remain untouched.
- Identifiers and times are retained for auditing but excluded from every predictive matrix.
- If only one patient and one recording are available, chronological blocks are used; this is weaker evidence and is reported in `preparation_report.json`.
- No performance result is claimed until the corresponding model notebook has completed.

In [ ]:
from eeg_app.src.experiments import prepare_balanced_exports
exported = prepare_balanced_exports(ROOT, seed=42)
print(f'Created {len(exported)} duration/balancing folders with whole/train/validation/test CSV files.')
print('\n'.join(str(path) for path in exported.values()))